# Foundation Model Benchmark: S&P 500 Daily Directional Prediction

This notebook benchmarks modern time series foundation models against the xLSTM-TS paper's results
([arXiv:2408.12408](https://arxiv.org/abs/2408.12408)) on the same S&P 500 daily close price dataset.

**Baseline from the paper's own code (denoised data):**
- xLSTM-TS: 64-66% test accuracy (varies across runs, no fixed seed)
- TiDE: 64.86%
- TCN: 63.41%
- DeepTCN: 63.77%

**Models tested here:**
1. **Chronos-2** (Amazon, 120M) - current GIFT-Eval benchmark leader
2. **TimesFM 2.5** (Google, 200M) - strong zero-shot forecaster
3. **TiRex** (NX-AI, 35M) - xLSTM-based foundation model, NeurIPS 2025
4. **Moirai 2.0** (Salesforce, 11M) - efficient decoder-only model

All models run **zero-shot** (no training on this data) and predict the next day's close price.
Directional accuracy is computed the same way as the original paper.

## Setup

In [ ]:
# Install dependencies (run once)
# Uncomment the models you want to test

!pip install pandas numpy scikit-learn matplotlib seaborn tqdm

# Chronos-2
!pip install chronos-forecasting torch

# TimesFM 2.5
!pip install timesfm

# TiRex
!pip install tirex-forecasting

# Moirai 2.0
!pip install uni2ts einops huggingface_hub

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, root_mean_squared_error,
    mean_absolute_percentage_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Detect device
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Using device: {DEVICE}')

## Constants

Same dataset and splits as the original paper.

In [ ]:
# Dataset
FILE_NAME = 'sp500_daily'
STOCK = 'S&P 500'

# Date splits (same as paper)
TRAIN_END_DATE = '2021-01-01'
VAL_END_DATE = '2022-07-01'

# Context length for foundation models
CONTEXT_LENGTH = 150  # same as xLSTM-TS sequence length

# Prediction horizon
PREDICTION_LENGTH = 1  # 1-day ahead

## Load Data

In [ ]:
# Load the CSV
file_path = os.path.join('..', 'data', 'datasets', FILE_NAME + '.csv')
df = pd.read_csv(file_path, header=0, index_col='Date')
df.index = pd.to_datetime(df.index, utc=True).normalize().tz_localize(None)

# Keep only Close price
close = df[['Close']].copy()
print(f'Total rows: {len(close)}')
print(f'Date range: {close.index[0]} to {close.index[-1]}')
close.head()

In [ ]:
# Split into train / val / test (same as paper)
train_end = pd.Timestamp(TRAIN_END_DATE)
val_end = pd.Timestamp(VAL_END_DATE)

train_df = close[close.index < train_end]
val_df = close[(close.index >= train_end) & (close.index < val_end)]
test_df = close[close.index >= val_end]

print(f'Train: {len(train_df)} rows ({train_df.index[0]} to {train_df.index[-1]})')
print(f'Val:   {len(val_df)} rows ({val_df.index[0]} to {val_df.index[-1]})')
print(f'Test:  {len(test_df)} rows ({test_df.index[0]} to {test_df.index[-1]})')

## Metrics

Same metrics as the original paper: regression (MAE, MSE, RMSE, RMSSE, MAPE, MASE, R2) and directional classification (accuracy, recall, precision, F1).

In [ ]:
def naive_forecasting(actual, seasonality=1):
    return actual[:-seasonality]

def rmsse(actual, predicted, seasonality=1):
    q = mean_squared_error(actual, predicted) / mean_squared_error(
        actual[seasonality:], naive_forecasting(actual, seasonality))
    return np.sqrt(q)

def mase(actual, predicted, seasonality=1):
    return mean_absolute_error(actual, predicted) / mean_absolute_error(
        actual[seasonality:], naive_forecasting(actual, seasonality))

def calculate_regression_metrics(actual, predicted):
    return {
        'MAE': mean_absolute_error(actual, predicted),
        'MSE': mean_squared_error(actual, predicted),
        'RMSE': root_mean_squared_error(actual, predicted),
        'RMSSE': rmsse(actual, predicted),
        'MAPE': mean_absolute_percentage_error(actual, predicted) * 100,
        'MASE': mase(actual, predicted),
        'R2': r2_score(actual, predicted),
    }

def calculate_directional_metrics(actual, predicted):
    """Compute directional accuracy from price series (same as paper)."""
    actual_dirs = (np.diff(actual.squeeze()) > 0).astype(int)
    pred_dirs = (np.diff(predicted.squeeze()) > 0).astype(int)
    return {
        'Test Accuracy': accuracy_score(actual_dirs, pred_dirs) * 100,
        'Recall': recall_score(actual_dirs, pred_dirs, pos_label=1) * 100,
        'Precision (Rise)': precision_score(actual_dirs, pred_dirs, pos_label=1) * 100,
        'Precision (Fall)': precision_score(actual_dirs, pred_dirs, pos_label=0) * 100,
        'F1 Score': f1_score(actual_dirs, pred_dirs, pos_label=1) * 100,
    }

def evaluate_model(actual, predicted, model_name):
    """Full evaluation: regression + directional metrics."""
    reg = calculate_regression_metrics(actual, predicted)
    dir_metrics = calculate_directional_metrics(actual, predicted)
    all_metrics = {**reg, **dir_metrics}
    print(f'\n--- {model_name} ---')
    for k, v in all_metrics.items():
        fmt = f'{v:.2f}%' if k in ('MAPE', 'Test Accuracy', 'Recall', 'Precision (Rise)', 'Precision (Fall)', 'F1 Score') else f'{v:.2f}'
        print(f'  {k}: {fmt}')
    return all_metrics

## Rolling Forecast Helper

Foundation models do zero-shot forecasting: given a context window of historical prices, predict the next value. We roll this across the test set one step at a time.

In [ ]:
# Prepare the full series for rolling prediction
# We need CONTEXT_LENGTH points before each test point
full_close = close['Close'].values

# Find the index where the test set starts
test_start_idx = len(train_df) + len(val_df)
test_actuals = full_close[test_start_idx:]
test_dates = close.index[test_start_idx:]

print(f'Test set: {len(test_actuals)} points')
print(f'Each prediction uses {CONTEXT_LENGTH} prior points as context')

---
## Model 1: Chronos-2 (Amazon, 120M)

Current benchmark leader on GIFT-Eval. Encoder-only transformer with group attention.

In [ ]:
from chronos import ChronosPipeline

chronos_model = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-base',
    device_map=DEVICE,
    torch_dtype=torch.float32,
)

chronos_preds = []
for i in tqdm(range(len(test_actuals)), desc='Chronos-2'):
    ctx_start = test_start_idx + i - CONTEXT_LENGTH
    context = torch.tensor(full_close[ctx_start:test_start_idx + i], dtype=torch.float32)
    forecast = chronos_model.predict(context.unsqueeze(0), prediction_length=PREDICTION_LENGTH)
    # forecast shape: (1, num_samples, prediction_length) - take median
    chronos_preds.append(forecast.median(dim=1).squeeze().item())

chronos_preds = np.array(chronos_preds)
chronos_metrics = evaluate_model(test_actuals, chronos_preds, 'Chronos-2')

---
## Model 2: TimesFM 2.5 (Google, 200M)

Decoder-only transformer with patch-based tokenization. Strong zero-shot performance.

In [ ]:
import timesfm

tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend='gpu' if DEVICE == 'cuda' else 'cpu',
        per_core_batch_size=1,
        horizon_len=PREDICTION_LENGTH,
        input_patch_len=32,
        output_patch_len=128,
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id='google/timesfm-2.0-200m-pytorch',
    ),
)

timesfm_preds = []
for i in tqdm(range(len(test_actuals)), desc='TimesFM'):
    ctx_start = test_start_idx + i - CONTEXT_LENGTH
    context = full_close[ctx_start:test_start_idx + i].tolist()
    point_forecast, _ = tfm.forecast([context])
    timesfm_preds.append(point_forecast[0][0])

timesfm_preds = np.array(timesfm_preds)
timesfm_metrics = evaluate_model(test_actuals, timesfm_preds, 'TimesFM 2.5')

---
## Model 3: TiRex (NX-AI, 35M)

xLSTM-based foundation model. NeurIPS 2025. Beats models 10x its size.

In [ ]:
from tirex import TiRexPipeline

tirex_model = TiRexPipeline.from_pretrained(
    'NX-AI/TiRex',
    device_map=DEVICE,
)

tirex_preds = []
for i in tqdm(range(len(test_actuals)), desc='TiRex'):
    ctx_start = test_start_idx + i - CONTEXT_LENGTH
    context = torch.tensor(full_close[ctx_start:test_start_idx + i], dtype=torch.float32)
    forecast = tirex_model.predict(context.unsqueeze(0), prediction_length=PREDICTION_LENGTH)
    tirex_preds.append(forecast.median(dim=1).squeeze().item())

tirex_preds = np.array(tirex_preds)
tirex_metrics = evaluate_model(test_actuals, tirex_preds, 'TiRex')

---
## Model 4: Moirai 2.0 (Salesforce, 11M)

Decoder-only transformer. 96% smaller than v1 with competitive accuracy.

In [ ]:
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

moirai_module = MoiraiModule.from_pretrained('Salesforce/moirai-2.0-R-small')

moirai_preds = []
for i in tqdm(range(len(test_actuals)), desc='Moirai 2.0'):
    ctx_start = test_start_idx + i - CONTEXT_LENGTH
    context_vals = full_close[ctx_start:test_start_idx + i]

    # Moirai expects a pandas DataFrame
    ctx_df = pd.DataFrame({
        'target': context_vals,
    }, index=pd.RangeIndex(len(context_vals)))

    predictor = moirai_module.create_predictor(
        prediction_length=PREDICTION_LENGTH,
        context_length=CONTEXT_LENGTH,
        num_samples=20,
    )

    # Use GluonTS-style dataset
    from gluonts.dataset.pandas import PandasDataset
    ds = PandasDataset.from_long_dataframe(ctx_df.reset_index(), target='target', item_id='index')

    forecasts = list(predictor.predict(ds))
    moirai_preds.append(np.median(forecasts[0].samples[:, 0]))

moirai_preds = np.array(moirai_preds)
moirai_metrics = evaluate_model(test_actuals, moirai_preds, 'Moirai 2.0')

---
## Results Comparison

In [ ]:
# Collect all foundation model results (only include models that ran successfully)
fm_results = {}
for name, var in [('Chronos-2', 'chronos_metrics'), ('TimesFM 2.5', 'timesfm_metrics'),
                   ('TiRex', 'tirex_metrics'), ('Moirai 2.0', 'moirai_metrics')]:
    try:
        metrics = eval(var)
        if metrics is not None:
            fm_results[name] = metrics
    except NameError:
        pass  # model wasn't run

# --------------------------------------------------------------------------
# Paper baselines from their GitHub notebook (exact reproduced numbers)
# --------------------------------------------------------------------------

# ORIGINAL data (no denoising) — fair comparison for zero-shot foundation models
paper_original = {
    'xLSTM-TS':  {'MAE': 38.25, 'MSE': 2325.60, 'RMSE': 48.22, 'RMSSE': 1.11, 'MAPE': 0.94, 'MASE': 1.16, 'R2': 0.97,
                   'Test Accuracy': 49.47, 'Recall': 53.68, 'Precision (Rise)': 50.00, 'Precision (Fall)': 48.84, 'F1 Score': 51.78,
                   'Validation Accuracy': 49.87, 'Train Accuracy': 48.10},
    'TSMixer':   {'MAE': 90.66, 'MSE': 15021.84, 'RMSE': 122.56, 'RMSSE': 3.38, 'MAPE': 2.15, 'MASE': 3.19, 'R2': 0.75,
                   'Test Accuracy': 46.38, 'Recall': 47.26, 'Precision (Rise)': 49.29, 'Precision (Fall)': 43.38, 'F1 Score': 48.25,
                   'Validation Accuracy': 56.00, 'Train Accuracy': 48.91},
    'N-HiTS':    {'MAE': 45.41, 'MSE': 3404.14, 'RMSE': 58.34, 'RMSSE': 1.61, 'MAPE': 1.08, 'MASE': 1.60, 'R2': 0.94,
                   'Test Accuracy': 50.72, 'Recall': 52.74, 'Precision (Rise)': 53.47, 'Precision (Fall)': 47.73, 'F1 Score': 53.10,
                   'Validation Accuracy': 57.45, 'Train Accuracy': 48.68},
    'TiDE':      {'MAE': 31.43, 'MSE': 1512.92, 'RMSE': 38.90, 'RMSSE': 1.07, 'MAPE': 0.75, 'MASE': 1.11, 'R2': 0.97,
                   'Test Accuracy': 51.45, 'Recall': 56.16, 'Precision (Rise)': 53.95, 'Precision (Fall)': 48.39, 'F1 Score': 55.03,
                   'Validation Accuracy': 53.82, 'Train Accuracy': 53.39},
    'TFT':       {'MAE': 52.56, 'MSE': 4924.10, 'RMSE': 70.17, 'RMSSE': 1.93, 'MAPE': 1.25, 'MASE': 1.85, 'R2': 0.92,
                   'Test Accuracy': 51.09, 'Recall': 52.74, 'Precision (Rise)': 53.85, 'Precision (Fall)': 48.12, 'F1 Score': 53.29,
                   'Validation Accuracy': 59.27, 'Train Accuracy': 48.31},
    'N-BEATS':   {'MAE': 50.18, 'MSE': 4029.71, 'RMSE': 63.48, 'RMSSE': 1.75, 'MAPE': 1.19, 'MASE': 1.77, 'R2': 0.93,
                   'Test Accuracy': 48.91, 'Recall': 48.63, 'Precision (Rise)': 51.82, 'Precision (Fall)': 46.04, 'F1 Score': 50.18,
                   'Validation Accuracy': 57.82, 'Train Accuracy': 50.03},
    'DeepTCN':   {'MAE': 50.08, 'MSE': 3471.25, 'RMSE': 58.92, 'RMSSE': 1.62, 'MAPE': 1.18, 'MASE': 1.76, 'R2': 0.94,
                   'Test Accuracy': 48.55, 'Recall': 51.37, 'Precision (Rise)': 51.37, 'Precision (Fall)': 45.38, 'F1 Score': 51.37,
                   'Validation Accuracy': 50.55, 'Train Accuracy': 47.06},
    'TCN':       {'MAE': 29.03, 'MSE': 1336.56, 'RMSE': 36.56, 'RMSSE': 1.01, 'MAPE': 0.69, 'MASE': 1.02, 'R2': 0.98,
                   'Test Accuracy': 49.64, 'Recall': 53.42, 'Precision (Rise)': 52.35, 'Precision (Fall)': 46.46, 'F1 Score': 52.88,
                   'Validation Accuracy': 49.09, 'Train Accuracy': 47.83},
}

# DENOISED data (wavelet denoised training) — upper bound from the paper
paper_denoised = {
    'xLSTM-TS':  {'MAE': 55.84, 'MSE': 4600.42, 'RMSE': 67.83, 'RMSSE': 1.56, 'MAPE': 1.37, 'MASE': 1.69, 'R2': 0.94,
                   'Test Accuracy': 66.22, 'Recall': 72.11, 'Precision (Rise)': 64.93, 'Precision (Fall)': 67.88, 'F1 Score': 68.33,
                   'Validation Accuracy': 64.80, 'Train Accuracy': 64.21},
    'TSMixer':   {'MAE': 85.58, 'MSE': 13163.98, 'RMSE': 114.73, 'RMSSE': 3.16, 'MAPE': 2.03, 'MASE': 3.02, 'R2': 0.78,
                   'Test Accuracy': 47.83, 'Recall': 49.32, 'Precision (Rise)': 50.70, 'Precision (Fall)': 44.78, 'F1 Score': 50.00,
                   'Validation Accuracy': 56.36, 'Train Accuracy': 49.80},
    'N-HiTS':    {'MAE': 33.96, 'MSE': 1835.65, 'RMSE': 42.84, 'RMSSE': 1.18, 'MAPE': 0.81, 'MASE': 1.20, 'R2': 0.97,
                   'Test Accuracy': 56.52, 'Recall': 56.85, 'Precision (Rise)': 59.29, 'Precision (Fall)': 53.68, 'F1 Score': 58.04,
                   'Validation Accuracy': 64.36, 'Train Accuracy': 56.38},
    'TiDE':      {'MAE': 23.31, 'MSE': 865.17, 'RMSE': 29.41, 'RMSSE': 0.81, 'MAPE': 0.56, 'MASE': 0.82, 'R2': 0.99,
                   'Test Accuracy': 64.86, 'Recall': 67.81, 'Precision (Rise)': 66.44, 'Precision (Fall)': 62.99, 'F1 Score': 67.12,
                   'Validation Accuracy': 68.36, 'Train Accuracy': 67.55},
    'TFT':       {'MAE': 47.97, 'MSE': 4100.21, 'RMSE': 64.03, 'RMSSE': 1.77, 'MAPE': 1.14, 'MASE': 1.69, 'R2': 0.93,
                   'Test Accuracy': 53.26, 'Recall': 54.79, 'Precision (Rise)': 55.94, 'Precision (Fall)': 50.38, 'F1 Score': 55.36,
                   'Validation Accuracy': 60.36, 'Train Accuracy': 52.02},
    'N-BEATS':   {'MAE': 32.14, 'MSE': 1666.18, 'RMSE': 40.82, 'RMSSE': 1.13, 'MAPE': 0.76, 'MASE': 1.13, 'R2': 0.97,
                   'Test Accuracy': 59.78, 'Recall': 62.33, 'Precision (Rise)': 61.90, 'Precision (Fall)': 57.36, 'F1 Score': 62.12,
                   'Validation Accuracy': 63.64, 'Train Accuracy': 58.56},
    'DeepTCN':   {'MAE': 54.97, 'MSE': 3924.06, 'RMSE': 62.64, 'RMSSE': 1.73, 'MAPE': 1.29, 'MASE': 1.94, 'R2': 0.93,
                   'Test Accuracy': 63.77, 'Recall': 71.23, 'Precision (Rise)': 64.20, 'Precision (Fall)': 63.16, 'F1 Score': 67.53,
                   'Validation Accuracy': 65.45, 'Train Accuracy': 64.65},
    'TCN':       {'MAE': 30.01, 'MSE': 1333.10, 'RMSE': 36.51, 'RMSSE': 1.01, 'MAPE': 0.71, 'MASE': 1.06, 'R2': 0.98,
                   'Test Accuracy': 63.41, 'Recall': 69.86, 'Precision (Rise)': 64.15, 'Precision (Fall)': 62.39, 'F1 Score': 66.89,
                   'Validation Accuracy': 65.82, 'Train Accuracy': 64.23},
}

# Format helper (same as paper)
def format_results(results_dict, title):
    df = pd.DataFrame(results_dict).T
    df = df.round(2)
    display = df.copy()
    pct_cols = ['MAPE', 'Test Accuracy', 'Recall', 'Precision (Rise)', 'Precision (Fall)',
                'F1 Score', 'Validation Accuracy', 'Train Accuracy']
    for col in pct_cols:
        if col in display.columns:
            display[col] = display[col].apply(lambda x: f'{x:.2f}%')
    print(f'\n{"="*80}')
    print(f'  {title}')
    print(f'{"="*80}')
    return df, display

# --------------------------------------------------------------------------
# Display all three tables
# --------------------------------------------------------------------------

# 1. Foundation models (zero-shot, raw data)
fm_df, fm_display = format_results(fm_results, 'FOUNDATION MODELS (zero-shot, no training, raw data)')
fm_display

In [ ]:
# 2. Paper baselines — original data (fair comparison: trained on raw data)
orig_df, orig_display = format_results(paper_original, 'PAPER BASELINES — ORIGINAL DATA (trained, no denoising)')
orig_display

In [ ]:
# 3. Paper baselines — denoised data (best paper results, uses wavelet preprocessing)
den_df, den_display = format_results(paper_denoised, 'PAPER BASELINES — DENOISED DATA (trained, wavelet denoised)')
den_display

In [ ]:
# --------------------------------------------------------------------------
# Visual comparison: all three groups side by side
# --------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Combine for plotting: foundation models + paper original + paper denoised (best only)
plot_data = {}
for name, row in fm_df.iterrows():
    plot_data[f'{name} (zero-shot)'] = {'Test Accuracy': row['Test Accuracy'], 'MAE': row['MAE'], 'group': 'Foundation (zero-shot)'}
for name, row in orig_df.iterrows():
    plot_data[f'{name} (original)'] = {'Test Accuracy': row['Test Accuracy'], 'MAE': row['MAE'], 'group': 'Paper (original data)'}
for name, row in den_df.iterrows():
    plot_data[f'{name} (denoised)'] = {'Test Accuracy': row['Test Accuracy'], 'MAE': row['MAE'], 'group': 'Paper (denoised data)'}

plot_df = pd.DataFrame(plot_data).T
plot_df = plot_df.sort_values('Test Accuracy', ascending=True)

# Color by group
color_map = {'Foundation (zero-shot)': '#FF5722', 'Paper (original data)': '#9E9E9E', 'Paper (denoised data)': '#2196F3'}
colors = [color_map[plot_df.loc[name, 'group']] for name in plot_df.index]

# Directional accuracy
axes[0].barh(plot_df.index, plot_df['Test Accuracy'].astype(float), color=colors)
axes[0].axvline(x=50, color='black', linestyle='--', alpha=0.5, label='Coin flip (50%)')
axes[0].set_xlabel('Test Accuracy (%)', fontsize=12)
axes[0].set_title('Directional Prediction Accuracy', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].tick_params(axis='y', labelsize=9)

# MAE
plot_df_mae = plot_df.sort_values('MAE', ascending=False)
colors_mae = [color_map[plot_df_mae.loc[name, 'group']] for name in plot_df_mae.index]
axes[1].barh(plot_df_mae.index, plot_df_mae['MAE'].astype(float), color=colors_mae)
axes[1].set_xlabel('MAE (lower is better)', fontsize=12)
axes[1].set_title('Mean Absolute Error', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='y', labelsize=9)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#FF5722', label='Foundation models (zero-shot)'),
                   Patch(facecolor='#9E9E9E', label='Paper baselines (original data)'),
                   Patch(facecolor='#2196F3', label='Paper baselines (denoised data)')]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=11, bbox_to_anchor=(0.5, -0.02))

plt.suptitle(f'{STOCK} Daily Close Price - Foundation Models vs xLSTM-TS Paper', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(bottom=0.08)
plt.show()

---
## Notes

**Important context for interpreting results:**

- The paper baselines were **trained** on this specific dataset (with wavelet denoising). Foundation models are running **zero-shot** with no training.
- If a foundation model matches or beats ~65% accuracy zero-shot, that's remarkable — it means pre-training on diverse time series data transfers to stock prediction without any task-specific optimization.
- Foundation models that underperform zero-shot can potentially be **fine-tuned** on this data for better results.
- The paper's xLSTM-TS results are non-deterministic (no fixed seed). Their GitHub notebook shows 66.22%, not the 71.28% claimed in the paper.